In [ ]:
import os
import shutil

if os.path.exists("logs"):
    shutil.rmtree("logs")
    print("Logs directory cleared")
else:
    print("No logs directory found")

No logs directory found


In [1]:
from utils import start_tensorboard

start_tensorboard()

📊 TensorBoard Setup Instructions

1. Open a terminal in Azure ML Studio

2. Run this command:

   tensorboard --logdir c:\Users\yanni\Dokumente\DSE\dse-sem2\CO2\MachineLearning\lecture-notes-CO2-2026\exercise_2\logs --host 0.0.0.0 --port 6006

3. Open this URL in your browser:

   https://Yannick-6006.swedencentral.instances.azureml.ms



'https://Yannick-6006.swedencentral.instances.azureml.ms'

# Introduction to Keras and Layer types

Previously you built a simple artificial neural net (ANN) with one hidden layer and sigmoid activation function. Now we will take a look at a few adaptations of such a basic net both in terms of NN architecture as well as layer types. We will still stay with Fully Connected Neural Networks for this first exercise!

things to go through:

- activation functions (issue with sigmoid -> ReLU, Leaky ReLU)
- drop out
- skip connections
- deeper neural nets
- potential issues of Fully Connected Neural Networks (Scaling?)
- Keras things like functional API


## Setup things

Make sure your environment has all the required packages available. Take care to have

- keras (Read [This short guide](https://keras.io/getting_started/))
- a backend of your choosing for Keras (I dont care which one you use but we will stay with Tensorflow for now)
- scikit-learn
- pandas
- tensorboard (optional)

Feel free to use a package manager of your choice (avoid conda) or a premade environment in Azure/Colab.
Also make sure you have the data files you need ready, I recommend putting them into a ./data/ subdir

## Loading and preparing the dataset


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction
df = pd.read_csv("data/energydata_complete.csv")

# For timeseries such as this, are rows really independent?
# Is there a separate approach to this problem? What patterns do we lose by treating the dataset like this?
df["date_time"] = pd.to_datetime(df["date"])

df["dayinweek"] = df["date_time"].dt.day_of_week
df["month"] = df["date_time"].dt.month
df["hour"] = df["date_time"].dt.hour
df["minutes"] = df["date_time"].dt.minute
df.drop(columns=["date_time"], inplace=True)

# how do we keep it random but reproducible?
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["Appliances", "date"]), df["Appliances"], random_state=42
)
print(X_train.shape, X_test.shape)

sd = StandardScaler()
X_train_scaled = sd.fit_transform(X_train)
X_test_scaled = sd.transform(X_test)
X_train.head(3)

(14801, 31) (4934, 31)


,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,...,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2,dayinweek,month,hour,minutes
8242,0,21.00,38.163333,18.50000,40.326667,20.230,38.00,19.926667,34.526667,18.730000,...,75.333333,4.666667,40.000000,0.600000,6.455464,6.455464,1,3,22,40
10603,0,21.00,40.450000,18.39000,44.090000,22.000,38.29,19.856667,38.863333,19.323333,...,97.000000,6.000000,50.833333,5.916667,6.797277,6.797277,4,3,8,10
18910,0,24.69,49.476000,24.32973,47.159009,26.834,43.84,23.997297,46.901351,22.927027,...,82.333333,2.000000,40.000000,14.433333,8.524160,8.524160,6,5,0,40


## Vanishing Gradient Problem with Sigmoid

Sigmoid's derivative ranges from 0 to 0.25 (max at the middle).

During backpropagation, gradients multiply layer-by-layer: 0.25 × 0.25 × 0.25...

**Result:** Gradients shrink exponentially in deeper networks → early layers barely learn. **Deeper Networks fail with sigmoid**

**Solution:** Use e.g. ReLU (derivative = 1 for positive inputs).


In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras

print(f"Using Keras {keras.__version__} with backend: {keras.backend.backend()}")

Using Keras 3.12.0 with backend: tensorflow


In [ ]:
import keras
from utils import eval_model, train_model

model_sigmoid = keras.models.Sequential(
    [
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(512, activation="sigmoid"),
        keras.layers.Dense(256, activation="sigmoid"),
        keras.layers.Dense(128, activation="sigmoid"),
        keras.layers.Dense(64, activation="sigmoid"),
        keras.layers.Dense(32, activation="sigmoid"),
        keras.layers.Dense(1, activation=None),
    ]
)

model_sigmoid.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_sigmoid.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model_relu = keras.models.Sequential(
    [
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(1, activation=None),
    ]
)

model_relu.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_relu.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history_sigmoid = train_model(model_sigmoid, X_train_scaled, y_train, "sigmoid_deep")

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 18976.1035 - mae: 90.6584 - val_loss: 17917.8730 - val_mae: 86.3037
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 17690.1426 - mae: 83.2031 - val_loss: 16750.6914 - val_mae: 79.2538
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 16555.1680 - mae: 76.1596 - val_loss: 15703.3857 - val_mae: 72.5127
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 15583.5244 - mae: 69.7023 - val_loss: 14805.8760 - val_mae: 66.3448
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 14743.0039 - mae: 63.9873 - val_loss: 14031.6338 - val_mae: 60.7770
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 14013.1367 - mae: 58.8595 - val_loss: 13354.5098 - val_mae: 56.2259
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 13382.9932 - mae: 54.7801 - val_loss: 12772.5713 - val_mae: 52.2547
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 12844.2334 - mae: 52.0651 - val_loss: 12280.

In [8]:
history_relu = train_model(model_relu, X_train_scaled, y_train, "relu_deep")

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 10221.3848 - mae: 56.5664 - val_loss: 8750.6953 - val_mae: 59.2184
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8891.0088 - mae: 52.1041 - val_loss: 8260.3877 - val_mae: 46.4422
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8456.7422 - mae: 50.5719 - val_loss: 7783.7485 - val_mae: 48.7716
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8113.8047 - mae: 48.8080 - val_loss: 7982.3369 - val_mae: 43.6303
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7808.4224 - mae: 47.1790 - val_loss: 7802.3071 - val_mae: 54.0468
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7644.3716 - mae: 46.8177 - val_loss: 7370.3818 - val_mae: 45.3261
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7355.0444 - mae: 45.9220 - val_loss: 7421.2720 - val_mae: 42.5466
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7071.2905 - mae: 44.5436 - val_loss: 8759.9990 - val_mae:

In [ ]:
model_leaky_relu = keras.models.Sequential(
    [
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),  #
        keras.layers.Dense(512),
        keras.layers.LeakyReLU(negative_slope=0.01),
        keras.layers.Dense(256),
        keras.layers.LeakyReLU(negative_slope=0.01),
        keras.layers.Dense(128),
        keras.layers.LeakyReLU(negative_slope=0.01),
        keras.layers.Dense(64),
        keras.layers.LeakyReLU(negative_slope=0.01),
        keras.layers.Dense(32),
        keras.layers.LeakyReLU(negative_slope=0.01),
        keras.layers.Dense(1, activation=None),
    ]
)

model_leaky_relu.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_leaky_relu.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history_sigmoid = train_model(model_sigmoid, X_train_scaled, y_train, "sigmoid_deep")
history_relu = train_model(model_relu, X_train_scaled, y_train, "relu_deep")
history_leaky_relu = train_model(
    model_leaky_relu, X_train_scaled, y_train, "leaky_relu_deep"
)

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7982.4243 - mae: 46.0680 - val_loss: 7962.3667 - val_mae: 47.4857
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7934.6094 - mae: 45.9792 - val_loss: 7900.8330 - val_mae: 45.5580
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7859.0542 - mae: 45.8328 - val_loss: 7821.6270 - val_mae: 44.2069
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7808.9478 - mae: 45.5490 - val_loss: 7820.3628 - val_mae: 46.7364
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7723.9819 - mae: 45.2991 - val_loss: 7806.9126 - val_mae: 45.6350
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7692.3452 - mae: 45.3035 - val_loss: 7681.3955 - val_mae: 44.2876
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7753.0347 - mae: 45.6952 - val_loss: 7782.7212 - val_mae: 43.6862
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7560.3955 - mae: 44.7794 - val_loss: 7708.3687 - val_mae: 

In [ ]:
# eval on test set

eval_model(model_sigmoid, X_test_scaled, y_test, "Sigmoid Deep Network")
eval_model(model_relu, X_test_scaled, y_test, "ReLU Deep Network")
eval_model(model_leaky_relu, X_test_scaled, y_test, "Leaky ReLU Deep Network")


  Sigmoid Deep Network - Test Results
  Test Loss (MSE): 6,641.17
  Test MAE:        40.22
  R² Score:        0.3315 (33.15% variance explained)


  ReLU Deep Network - Test Results
  Test Loss (MSE): 6,773.55
  Test MAE:        39.02
  R² Score:        0.3181 (31.81% variance explained)


  Leaky ReLU Deep Network - Test Results
  Test Loss (MSE): 6,531.00
  Test MAE:        40.52
  R² Score:        0.3426 (34.26% variance explained)



(6531.00146484375, 40.51664733886719, 0.3425515294075012)

## Dropout - Preventing Overfitting

Deep networks can **memorize trained-on data** instead of learning generalizations.

**Dropout** randomly sets a percentage of neurons to 0 during each training step (e.g., 30% dropout → 30% of neurons turned off).

This forces the network to learn **robust features** that don't rely on a specific small subset of neurons. We will see on the tensorboard curves, that the model is learning slower but also less likely to overfit.


In [12]:
## Dropout
model_with_dropout = keras.models.Sequential(
    [
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation=None),
    ]
)

model_with_dropout.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_with_dropout.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
history_with_dropout = train_model(
    model_with_dropout, X_train_scaled, y_train, "with_dropout", epochs=50
)

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 10878.9072 - mae: 58.2087 - val_loss: 9655.6729 - val_mae: 46.4808
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9508.9727 - mae: 53.9151 - val_loss: 8409.7949 - val_mae: 57.8651
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9236.0811 - mae: 53.0088 - val_loss: 8128.1973 - val_mae: 50.5419
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9003.1621 - mae: 51.9113 - val_loss: 7820.1270 - val_mae: 48.6071
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8902.0322 - mae: 50.8758 - val_loss: 7811.0024 - val_mae: 43.3113
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8671.9023 - mae: 50.0978 - val_loss: 8535.9072 - val_mae: 42.0026
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8542.1787 - mae: 49.5098 - val_loss: 7767.1240 - val_mae: 54.3747
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8422.6631 - mae: 49.2536 - val_loss: 8187.6621 - val_mae:

In [14]:
eval_model(model_with_dropout, X_test_scaled, y_test, "ReLU with Dropout")


  ReLU with Dropout - Test Results
  Test Loss (MSE): 6,244.39
  Test MAE:        39.45
  R² Score:        0.3714 (37.14% variance explained)



(6244.39013671875, 39.4474983215332, 0.3714033365249634)

# Batchnorm

**Problem:** During training, the distribution of inputs to each layer changes as previous layers update (internal covariate shift), slowing down training.

**Batch Normalization** normalizes layer inputs by computing mean and standard deviation across the batch, then scaling and shifting with learned parameters.

Batchnorm is quite optional, at the end the effect on performance should be tested empirically.


In [15]:
model_with_batchnorm = keras.models.Sequential(
    [
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(512),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(128),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(32),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation=None),
    ]
)

model_with_batchnorm.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_with_batchnorm.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 194,945 (761.50 KB)

 Trainable params: 192,961 (753.75 KB)

 Non-trainable params: 1,984 (7.75 KB)

In [16]:
history_with_batchnorm = train_model(
    model_with_batchnorm,
    X_train_scaled,
    y_train,
    "with_dropout_and_batchnorm",
    epochs=50,
)

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 19135.5293 - mae: 93.5469 - val_loss: 16474.6680 - val_mae: 84.7812
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 15633.8213 - mae: 78.8685 - val_loss: 13362.6562 - val_mae: 71.7924
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 11751.4336 - mae: 60.5908 - val_loss: 9202.3496 - val_mae: 53.5555
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 9359.5352 - mae: 50.9547 - val_loss: 8072.6099 - val_mae: 46.9552
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 8460.8105 - mae: 48.6981 - val_loss: 7697.0073 - val_mae: 42.7139
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 8212.1455 - mae: 48.4192 - val_loss: 7567.6719 - val_mae: 44.5765
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 8211.0293 - mae: 48.7174 - val_loss: 7430.5938 - val_mae: 42.7924
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 8131.7271 - mae: 48.6496 - val_loss: 7363.5063 - val_

In [17]:
eval_model(
    model_with_batchnorm, X_test_scaled, y_test, "ReLU with Dropout and Batchnorm"
)


  ReLU with Dropout and Batchnorm - Test Results
  Test Loss (MSE): 6,173.86
  Test MAE:        39.08
  R² Score:        0.3785 (37.85% variance explained)



(6173.86376953125, 39.08433532714844, 0.3785029649734497)

## Keras Functional API

The **Sequential API** is simple but limited - layers stack linearly, no branching or multiple inputs/outputs. It is quite rigid in how we can define and interact with the architecture.

The **Functional API** is more flexible (and follows typical functional programming styles - think lambda):

- Multiple inputs/outputs
- Layer sharing
- Branching and merging
- Skip connections (residual networks)
- also it allows us to dynamically construct different architectures e.g. for hyperparam search.


In [18]:
# Functional API version of dropout model
inputs = keras.layers.Input(shape=(X_train_scaled.shape[1],))

x = keras.layers.Dense(512, activation="relu")(inputs)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(64, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

x = keras.layers.Dense(32, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

outputs = keras.layers.Dense(1, activation=None)(x)

model_functional = keras.Model(
    inputs=inputs, outputs=outputs, name="functional_dropout"
)
model_functional.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_functional.summary()

Model: "functional_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 31)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [2]:
# Your turn, implement a skip connection from the first dense+dropout layer block to the last (excluding output layer)
# Hint https://keras.io/api/layers/merging_layers/add/

In [20]:
inputs = keras.layers.Input(shape=(X_train_scaled.shape[1],))

x = keras.layers.Dense(256, activation="relu")(inputs)
x = keras.layers.Dropout(0.3)(x)
skip = x

x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

x = keras.layers.Add()([x, skip])

x = keras.layers.Dense(64, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

outputs = keras.layers.Dense(1, activation=None)(x)

model_skip = keras.Model(inputs=inputs, outputs=outputs, name="skip_connection")
model_skip.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_skip.summary()

Model: "skip_connection"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 31)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_36 (Dense)    │ (None, 256)       │      8,192 │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 256)       │          0 │ dense_36[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_37 (Dense)    │ (None, 256)       │     65,792 │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 256)       │          0 │ dense_37[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_38 (Dense)    │ (None, 256)       │     65,792 │ dropout_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 256)       │          0 │ dense_38[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_39 (Dense)    │ (None, 256)       │     65,792 │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 256)       │          0 │ dense_39[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256)       │          0 │ dropout_18[0][0], │
│                     │                   │            │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_40 (Dense)    │ (None, 64)        │     16,448 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 64)        │          0 │ dense_40[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_41 (Dense)    │ (None, 1)         │         65 │ dropout_19[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 222,081 (867.50 KB)

 Trainable params: 222,081 (867.50 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
history_skip = train_model(
    model_skip, X_train_scaled, y_train, "with_skip_connection", epochs=50
)

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 10422.0537 - mae: 56.7800 - val_loss: 8507.1797 - val_mae: 52.7121
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9280.4990 - mae: 53.3492 - val_loss: 8329.9248 - val_mae: 44.8581
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8951.7969 - mae: 51.0635 - val_loss: 8111.2622 - val_mae: 44.5722
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8690.2305 - mae: 50.2745 - val_loss: 7831.8306 - val_mae: 45.9984
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8407.6348 - mae: 49.0347 - val_loss: 8016.8091 - val_mae: 41.5586
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8341.6055 - mae: 48.3332 - val_loss: 7636.3926 - val_mae: 44.6904
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8148.2466 - mae: 47.8108 - val_loss: 7501.7197 - val_mae: 44.5691
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7950.3594 - mae: 47.2434 - val_loss: 8144.9692 - val_mae:

In [22]:
eval_model(model_skip, X_test_scaled, y_test, "Skip Connection Network")


  Skip Connection Network - Test Results
  Test Loss (MSE): 6,503.07
  Test MAE:        40.94
  R² Score:        0.3454 (34.54% variance explained)



(6503.07421875, 40.9405517578125, 0.34536266326904297)